# UGP Post Analysis

Notebook ini menjalankan dua bagian analisis:

- `CORE 1a`: perbandingan hasil `Hit@10` antara `Random`, `Profile-Based Partition`, dan `Single Attribute Partition`
- `CORE 1b`: analisis similarity berbasis geometri user-state mengikuti kerangka `diagnosis.py`

Semua output disimpan ke `D:/Bob_Skripsi_Do Not Delete/results_ugp_analysis/UGP_Post_Analysis`.


In [ ]:
from pathlib import Path
import copy
import hashlib
import math
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from scipy import stats
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'figure.figsize': (10, 5.8),
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.edgecolor': '#D0D7DE',
    'axes.grid': True,
    'grid.color': '#E5E7EB',
    'grid.linewidth': 0.8,
    'grid.alpha': 1.0,
    'axes.axisbelow': True,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 15,
    'axes.labelsize': 11,
    'legend.frameon': False,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Trebuchet MS', 'Verdana', 'DejaVu Sans']
})

SEED = 97620260313
K_TARGET = 10
FORGET_PERCENTAGE = 1
MAX_RETAIN_DROP_PP = 5.0
TARGET_METHODS = ['Ye_multi', 'New_True_inf', 'Gradient_Ascent']
METHOD_LABELS = {
    'Ye_multi': 'Metode Terdahulu (Multi Env)',
    'New_True_inf': 'Metode diusulkan',
    'Gradient_Ascent': 'Gradient Ascent',
}
METHOD_COLORS = {
    'Ye_multi': '#EAB308',
    'New_True_inf': '#2563EB',
    'Gradient_Ascent': '#7C3AED',
}
PARTITION_COLORS = {
    'Random': '#2563EB',
    'Profile-Based Partition': '#DC2626',
    'Single Attribute Partition': '#059669',
}
ATTRIBUTE_COLORS = {
    'Gender': '#7C3AED',
    'Age': '#2563EB',
    'Occupation': '#EA580C',
}
AGE_LABELS = {
    1: 'Under 18',
    18: '18-24',
    25: '25-34',
    35: '35-44',
    45: '45-49',
    50: '50-55',
    56: '56+',
}
OCCUPATION_LABELS = {
    0: 'Other / not specified',
    1: 'Academic/Educator',
    2: 'Artist',
    3: 'Clerical/Admin',
    4: 'College/Grad Student',
    5: 'Customer Service',
    6: 'Doctor/Health Care',
    7: 'Executive/Managerial',
    8: 'Farmer',
    9: 'Homemaker',
    10: 'K-12 Student',
    11: 'Lawyer',
    12: 'Programmer',
    13: 'Retired',
    14: 'Sales/Marketing',
    15: 'Scientist',
    16: 'Self-Employed',
    17: 'Technician/Engineer',
    18: 'Tradesman/Craftsman',
    19: 'Unemployed',
    20: 'Writer',
}

def make_seed(*args):
    key = f"{SEED}|{'|'.join(str(a) for a in args)}".encode('utf-8')
    return int(hashlib.sha256(key).hexdigest()[:8], 16)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed % (2**31 - 1))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed % (2**31 - 1))

set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def find_repo_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / 'Raw').exists() and (candidate / 'Reinforcement-Unlearning-Movielens').exists():
            return candidate
    raise FileNotFoundError('Repo root tidak ditemukan dari working directory saat ini.')

def resolve_existing_path(candidates, description):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'{description} tidak ditemukan. Candidates: {candidates}')

def compute_drop_pp(base_col, after_col, frame):
    return (frame[base_col] - frame[after_col]) * 100.0

def format_setting_label(setting_type, raw_value, setting_label=None):
    if setting_type == 'gender':
        return 'Female' if str(raw_value) == 'F' else 'Male'
    if setting_type == 'age':
        return AGE_LABELS.get(int(raw_value), str(raw_value))
    if setting_type == 'occupation':
        occ = int(raw_value)
        return OCCUPATION_LABELS.get(occ, f'Occupation {occ}')
    if setting_label:
        return str(setting_label).replace('_', ' ').title()
    return str(raw_value)

def add_bar_labels(ax, bars, fmt='{:.2f}', suffix=''):
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + max(0.1, abs(height) * 0.015),
            f"{fmt.format(height)}{suffix}",
            ha='center', va='bottom', fontsize=10, color='#334155'
        )

UGP_METRICS_PATH = Path(r'D:/Bob_Skripsi_Do Not Delete') / 'results_ugp_analysis' / 'metrics' / 'relearn_metrics.csv'
RANDOM_RESULTS_PATH = Path(r'C:/Bob')  / 'results' / '1_percent' / 'tuning_full_results.csv'
PROFILE_RESULTS_PATH = Path(r'D:/Bob_Skripsi_Do Not Delete')  / 'Demography' / '1_percent' / 'tuning_full_results.csv'
OUTPUT_ROOT = Path(r'D:/Bob_Skripsi_Do Not Delete/results_ugp_analysis/UGP_Post_Analysis')
FIG_DIR = OUTPUT_ROOT / 'figures'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = resolve_existing_path([
    Path(r'C:/Bob/ml-1m'),
    Path(r'D:/Bob/ml-1m'),
    REPO_ROOT / 'data_movie',
], 'MovieLens data directory')

print(f'REPO_ROOT            : {REPO_ROOT}')
print(f'DATA_DIR             : {DATA_DIR}')
print(f'UGP_METRICS_PATH     : {UGP_METRICS_PATH}')
print(f'RANDOM_RESULTS_PATH  : {RANDOM_RESULTS_PATH}')
print(f'PROFILE_RESULTS_PATH : {PROFILE_RESULTS_PATH}')
print(f'OUTPUT_ROOT          : {OUTPUT_ROOT}')
print(f'DEVICE               : {DEVICE}')


In [ ]:
def load_partition_results(path, family_label):
    df = pd.read_csv(path).copy()
    df = df[df['method'].isin(TARGET_METHODS)].copy()
    df = df[df['K'] == K_TARGET].copy()
    numeric_cols = [
        'train_lr', 'gamma', 'hidden_dim', 'train_batch', 'unlearn_lr', 'unlearn_iters', 'lambda_retain',
        'retain_Hit', 'forget_Hit', 'base_retain_Hit', 'base_forget_Hit', 'retain_NDCG', 'forget_NDCG',
        'base_retain_NDCG', 'base_forget_NDCG'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['retain_drop_hit_pp'] = compute_drop_pp('base_retain_Hit', 'retain_Hit', df)
    df['forget_drop_hit_pp'] = compute_drop_pp('base_forget_Hit', 'forget_Hit', df)
    df['partition_family'] = family_label
    df['attribute_group'] = 'N/A'
    df['setting_type'] = family_label.lower().replace(' ', '_')
    df['setting_value_raw'] = FORGET_PERCENTAGE
    df['setting_label'] = f'{family_label} 1%'
    df['setting_display'] = f'{family_label} 1%'
    df['forget_user_count'] = np.nan
    return df

def pick_best_partition_rows(df):
    constrained = df[df['retain_drop_hit_pp'] <= MAX_RETAIN_DROP_PP].copy()
    constrained = constrained.sort_values(
        ['method', 'forget_drop_hit_pp', 'lambda_retain'],
        ascending=[True, False, True],
        kind='mergesort'
    )
    return constrained.groupby('method', as_index=False).first()

def load_ugp_metrics(path):
    df = pd.read_csv(path).copy()
    df = df[df['method'].isin(TARGET_METHODS)].copy()
    df = df[df['K'] == K_TARGET].copy()
    numeric_cols = [
        'forget_user_count', 'retain_user_count', 'retain_Hit', 'forget_Hit', 'base_retain_Hit', 'base_forget_Hit',
        'retain_NDCG', 'forget_NDCG', 'base_retain_NDCG', 'base_forget_NDCG', 'lambda_retain', 'hidden_dim',
        'train_batch', 'train_lr', 'gamma', 'unlearn_lr', 'unlearn_iters'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['retain_drop_hit_pp'] = compute_drop_pp('base_retain_Hit', 'retain_Hit', df)
    df['forget_drop_hit_pp'] = compute_drop_pp('base_forget_Hit', 'forget_Hit', df)
    df['partition_family'] = 'Single Attribute Partition'
    group_map = {'gender': 'Gender', 'age': 'Age', 'occupation': 'Occupation'}
    df['attribute_group'] = df['setting_type'].map(group_map)
    df['setting_display'] = [format_setting_label(st, rv, sl) for st, rv, sl in zip(df['setting_type'], df['setting_value_raw'], df['setting_label'])]
    return df

def load_data(data_dir):
    ratings_df = pd.read_csv(data_dir / 'ratings.dat', sep='::', engine='python', names=['user_id', 'movie_id', 'rating', 'timestamp'])
    movies_df = pd.read_csv(data_dir / 'movies.dat', sep='::', engine='python', names=['movie_id', 'title', 'genres'], encoding='ISO-8859-1')
    users_df = pd.read_csv(data_dir / 'users.dat', sep='::', engine='python', names=['user_id', 'gender', 'age', 'occupation', 'zip'])
    return ratings_df, movies_df, users_df

class PolicyNet(nn.Module):
    def __init__(self, state_dim, num_actions, hidden_dim=256):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.logits = nn.Linear(hidden_dim, num_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.logits(x)

def prepare_similarity_environment():
    ratings_df, movies_df, users_df = load_data(DATA_DIR)
    sample_users = ratings_df['user_id'].unique().tolist()
    pilot_ratings_all = ratings_df[ratings_df['user_id'].isin(sample_users)].copy()
    pilot_ratings_all.sort_values(['user_id', 'timestamp'], inplace=True)
    pilot_users_df = users_df[users_df['user_id'].isin(sample_users)]
    user_cats = pilot_users_df[['user_id', 'gender', 'age', 'occupation']].copy()
    oh = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    user_cat_mat = oh.fit_transform(user_cats[['gender', 'age', 'occupation']])
    user_feat_df = pd.DataFrame(user_cat_mat, index=user_cats['user_id'])
    all_genres = sorted({g for s in movies_df['genres'].astype(str) for g in s.split('|')})
    genre_to_idx = {g: i for i, g in enumerate(all_genres)}
    num_genres = len(all_genres)

    def movie_genre_vector(genres_str):
        v = np.zeros(num_genres, dtype=np.float32)
        for genre in str(genres_str).split('|'):
            if genre in genre_to_idx:
                v[genre_to_idx[genre]] = 1.0
        return v

    movies_df['genre_vec'] = movies_df['genres'].apply(movie_genre_vector)
    movie_genre_map = {mid: movies_df.loc[movies_df['movie_id'] == mid, 'genre_vec'].values[0] for mid in movies_df['movie_id'].unique()}
    candidate_movies = np.array(sorted(pilot_ratings_all['movie_id'].unique()))

    def build_state_fn(user_id, watched_movies):
        user_feat = user_feat_df.loc[user_id].values.astype(np.float32)
        pref_vec = np.zeros(num_genres, dtype=np.float32)
        for mid in watched_movies:
            if mid in movie_genre_map:
                pref_vec += movie_genre_map[mid]
        s = pref_vec.sum()
        if s > 0:
            pref_vec /= s
        return np.concatenate([user_feat, pref_vec]).astype(np.float32)

    trajectories_all = [
        {'user_id': uid, 'movies': g['movie_id'].tolist(), 'ratings': g['rating'].tolist()}
        for uid, g in pilot_ratings_all.groupby('user_id') if len(g) >= 5
    ]
    user_states = {}
    for traj in trajectories_all:
        midpoint = len(traj['movies']) // 2
        user_states[traj['user_id']] = build_state_fn(traj['user_id'], traj['movies'][:midpoint])
    return {
        'ratings_df': ratings_df,
        'movies_df': movies_df,
        'users_df': users_df,
        'sample_users': sample_users,
        'candidate_movies': candidate_movies,
        'trajectories_all': trajectories_all,
        'user_states': user_states,
        'state_dim': user_feat_df.shape[1] + num_genres,
        'build_state_fn': build_state_fn,
    }

def evaluate_per_user(policy_net, trajectories, candidate_movies, build_state_fn, K=10):
    rows = []
    candidate_movies = np.array(candidate_movies)
    policy_net.eval()
    for traj in trajectories:
        uid = traj['user_id']
        movies = traj['movies']
        if len(movies) < 5:
            continue
        midpoint = len(movies) // 2
        future = set(movies[midpoint:])
        state = build_state_fn(uid, movies[:midpoint])
        state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            probs = F.softmax(policy_net(state_t), dim=-1).squeeze(0).cpu().numpy()
        topk_movies = candidate_movies[np.argsort(-probs, kind='stable')[:K]]
        hit = float(any(m in future for m in topk_movies))
        rows.append({'uid': uid, 'hit': hit})
    return pd.DataFrame(rows)

def load_policy(model_path, state_dim, num_actions, hidden_dim):
    if not os.path.exists(model_path):
        raise FileNotFoundError(f'Model path tidak ditemukan: {model_path}')
    net = PolicyNet(state_dim, num_actions, hidden_dim=int(hidden_dim)).to(DEVICE)
    net.load_state_dict(torch.load(model_path, map_location=DEVICE))
    net.eval()
    return net

def build_random_split(sample_users):
    users = np.array(list(sample_users), dtype=int)
    set_seed(SEED)
    np.random.shuffle(users)
    split_amt = int(np.round(FORGET_PERCENTAGE / 100 * len(users)))
    return users[:split_amt], users[split_amt:]

def get_multiplier_demo(gender, age, occupation):
    mult = 1.0
    if gender == 'F':
        mult *= 1.36
    if age == 1:
        mult *= 1.0
    elif age == 18:
        mult *= 1.65
    elif age == 25:
        mult *= 1.60
    elif age == 35:
        mult *= 1.55
    elif age == 45:
        mult *= 1.22
    elif age == 50:
        mult *= 1.0
    elif age == 56:
        mult *= 0.92
    if occupation in (1, 4, 6, 10, 11, 15):
        mult *= 1.22
    return mult

def build_profile_split(users_df, sample_users):
    users_sorted = np.array(sorted(sample_users), dtype=int)
    users_meta = users_df[users_df['user_id'].isin(sample_users)].set_index('user_id')
    user_mult = []
    for uid in users_sorted:
        row = users_meta.loc[uid]
        user_mult.append((uid, get_multiplier_demo(row['gender'], row['age'], row['occupation'])))
    mean_mult = float(np.mean([m for _, m in user_mult]))
    base_prob = (FORGET_PERCENTAGE / 100) / mean_mult
    rng = np.random.default_rng(make_seed('forget_split'))
    forget_users, retain_users = [], []
    for uid, mult in user_mult:
        prob = min(base_prob * mult, 0.30)
        if rng.random() < prob:
            forget_users.append(uid)
        else:
            retain_users.append(uid)
    return np.array(forget_users, dtype=int), np.array(retain_users, dtype=int)

def build_single_attribute_split(users_df, sample_users, setting_type, setting_value_raw, target_count=60):
    users_meta = users_df[users_df['user_id'].isin(sample_users)].copy().set_index('user_id').sort_index()
    if setting_type == 'gender':
        subset = users_meta[users_meta['gender'] == str(setting_value_raw)]
    elif setting_type == 'age':
        subset = users_meta[users_meta['age'] == int(float(setting_value_raw))]
    elif setting_type == 'occupation':
        subset = users_meta[users_meta['occupation'] == int(float(setting_value_raw))]
    else:
        raise ValueError(f'Unsupported setting_type: {setting_type}')
    forget_users = subset.index.to_numpy(dtype=int)[:min(target_count, len(subset))]
    retain_users = np.array(sorted(set(sample_users) - set(forget_users.tolist())), dtype=int)
    return forget_users, retain_users

def compute_cosine_similarity_stats(df_retain, core_uids, retain_user_states, forget_user_states):
    if not core_uids:
        return []
    valid_core_uids = [uid for uid in core_uids if uid in forget_user_states]
    if not valid_core_uids:
        return []
    core_mat = np.stack([forget_user_states[uid] for uid in valid_core_uids])
    c_norm = core_mat / (np.linalg.norm(core_mat, axis=1, keepdims=True) + 1e-10)
    centroid = c_norm.mean(axis=0)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-10)

    work = df_retain.copy()
    centroid_vals = []
    pairwise_vals = []
    valid_mask = []
    for uid in work['uid']:
        if uid not in retain_user_states:
            centroid_vals.append(np.nan)
            pairwise_vals.append(np.nan)
            valid_mask.append(False)
            continue
        r_vec = retain_user_states[uid]
        r_norm = r_vec / (np.linalg.norm(r_vec) + 1e-10)
        centroid_vals.append(float(r_norm @ centroid))
        pairwise_vals.append(float(np.max(r_norm @ c_norm.T)))
        valid_mask.append(True)
    work['centroid_val'] = centroid_vals
    work['pairwise_val'] = pairwise_vals
    work = work[work['centroid_val'].notna()].copy()

    stats_rows = []
    for value_col, proximity_label in [('centroid_val', 'Centroid'), ('pairwise_val', 'Pairwise')]:
        damaged = work[work['damaged']][value_col].dropna().values
        ok = work[~work['damaged']][value_col].dropna().values
        if len(damaged) < 3 or len(ok) < 3:
            p_val = np.nan
            u_stat = np.nan
            corr_r = np.nan
            corr_p = np.nan
        else:
            u_stat, p_val = stats.mannwhitneyu(damaged, ok, alternative='greater')
            corr_r, corr_p = stats.pearsonr(work[value_col].values, work['delta'].values)
        stats_rows.append({
            'proximity_type': proximity_label,
            'N Damaged': int(len(damaged)),
            'N OK': int(len(ok)),
            'Mean (Damaged)': float(np.mean(damaged)) if len(damaged) else np.nan,
            'Mean (OK)': float(np.mean(ok)) if len(ok) else np.nan,
            'Delta Mean': float(np.mean(damaged) - np.mean(ok)) if len(damaged) and len(ok) else np.nan,
            'U Statistic': float(u_stat) if pd.notna(u_stat) else np.nan,
            'p-value': float(p_val) if pd.notna(p_val) else np.nan,
            'Pearson r': float(corr_r) if pd.notna(corr_r) else np.nan,
            'Pearson p': float(corr_p) if pd.notna(corr_p) else np.nan,
        })
    return stats_rows

print('Shared helpers ready.')


In [ ]:
random_results_k10 = load_partition_results(RANDOM_RESULTS_PATH, 'Random')
profile_results_k10 = load_partition_results(PROFILE_RESULTS_PATH, 'Profile-Based Partition')
ugp_results_k10 = load_ugp_metrics(UGP_METRICS_PATH)

selected_random_df = pick_best_partition_rows(random_results_k10)
selected_profile_df = pick_best_partition_rows(profile_results_k10)
selected_ugp_best_df = (
    ugp_results_k10
    .sort_values(['method', 'forget_drop_hit_pp', 'retain_drop_hit_pp', 'setting_label'], ascending=[True, False, True, True], kind='mergesort')
    .groupby('method', as_index=False)
    .first()
)

core1a_best_partition_comparison = pd.concat([
    selected_random_df[['partition_family', 'method', 'setting_display', 'forget_drop_hit_pp', 'retain_drop_hit_pp', 'forget_Hit', 'retain_Hit', 'trained_model_path', 'unlearned_model_path']].copy(),
    selected_profile_df[['partition_family', 'method', 'setting_display', 'forget_drop_hit_pp', 'retain_drop_hit_pp', 'forget_Hit', 'retain_Hit', 'trained_model_path', 'unlearned_model_path']].copy(),
    selected_ugp_best_df[['partition_family', 'method', 'setting_display', 'forget_drop_hit_pp', 'retain_drop_hit_pp', 'forget_Hit', 'retain_Hit', 'trained_model_path', 'unlearned_model_path', 'setting_type', 'setting_value_raw', 'setting_label']].copy(),
], ignore_index=True)
core1a_best_partition_comparison['method_label'] = core1a_best_partition_comparison['method'].map(METHOD_LABELS)
core1a_best_partition_comparison.to_csv(TABLE_DIR / 'core1a_best_partition_comparison.csv', index=False)

single_attribute_group_summary = (
    ugp_results_k10
    .groupby(['method', 'attribute_group'], as_index=False)
    .agg(
        mean_forget_drop_hit_pp=('forget_drop_hit_pp', 'mean'),
        mean_retain_drop_hit_pp=('retain_drop_hit_pp', 'mean'),
        mean_forget_Hit=('forget_Hit', 'mean'),
    )
)
best_within_group = (
    ugp_results_k10
    .sort_values(['method', 'attribute_group', 'forget_drop_hit_pp', 'retain_drop_hit_pp', 'setting_label'], ascending=[True, True, False, True, True], kind='mergesort')
    .groupby(['method', 'attribute_group'], as_index=False)
    .first()[['method', 'attribute_group', 'setting_display', 'setting_label', 'forget_user_count', 'forget_drop_hit_pp', 'retain_drop_hit_pp', 'forget_Hit']]
    .rename(columns={
        'setting_display': 'best_setting_display',
        'setting_label': 'best_setting_label',
        'forget_user_count': 'best_setting_forget_user_count',
        'forget_drop_hit_pp': 'best_setting_forget_drop_hit_pp',
        'retain_drop_hit_pp': 'best_setting_retain_drop_hit_pp',
        'forget_Hit': 'best_setting_forget_Hit',
    })
)
core1a_single_attribute_group_summary = single_attribute_group_summary.merge(best_within_group, on=['method', 'attribute_group'], how='left')
core1a_single_attribute_group_summary['method_label'] = core1a_single_attribute_group_summary['method'].map(METHOD_LABELS)
core1a_single_attribute_group_summary.to_csv(TABLE_DIR / 'core1a_single_attribute_group_summary.csv', index=False)

core1a_attribute_aggregate_summary = (
    ugp_results_k10
    .groupby('attribute_group', as_index=False)
    .agg(
        mean_forget_drop_hit_pp=('forget_drop_hit_pp', 'mean'),
        mean_retain_drop_hit_pp=('retain_drop_hit_pp', 'mean'),
        mean_forget_Hit=('forget_Hit', 'mean'),
    )
    .sort_values('mean_forget_drop_hit_pp', ascending=False)
    .reset_index(drop=True)
)
core1a_attribute_aggregate_summary.to_csv(TABLE_DIR / 'core1a_attribute_aggregate_summary.csv', index=False)

display(core1a_best_partition_comparison[['partition_family', 'method_label', 'setting_display', 'forget_drop_hit_pp', 'retain_drop_hit_pp', 'forget_Hit', 'retain_Hit']].sort_values(['method_label', 'partition_family']))
display(core1a_single_attribute_group_summary[['method_label', 'attribute_group', 'mean_forget_drop_hit_pp', 'mean_retain_drop_hit_pp', 'mean_forget_Hit', 'best_setting_display', 'best_setting_forget_drop_hit_pp']])
display(core1a_attribute_aggregate_summary)

fig, ax = plt.subplots(figsize=(11.5, 6.2))
method_order = TARGET_METHODS
family_order = ['Random', 'Profile-Based Partition', 'Single Attribute Partition']
x = np.arange(len(method_order))
width = 0.24
for idx, family in enumerate(family_order):
    family_df = core1a_best_partition_comparison[core1a_best_partition_comparison['partition_family'] == family].set_index('method').reindex(method_order).reset_index()
    offset = (idx - 1) * width
    bars = ax.bar(x + offset, family_df['forget_drop_hit_pp'], width=width, color=PARTITION_COLORS[family], label=family)
    add_bar_labels(ax, bars)
ax.set_title('CORE 1a - Best Forget Drop Hit@10 per Method', loc='left', pad=14)
ax.set_ylabel('Forget Drop Hit@10 (pp)')
ax.set_xticks(x)
ax.set_xticklabels([METHOD_LABELS[m] for m in method_order])
ax.set_ylim(0, core1a_best_partition_comparison['forget_drop_hit_pp'].max() * 1.18)
ax.legend(ncol=3, bbox_to_anchor=(0.5, 1.02), loc='lower center')
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(FIG_DIR / 'core1a_best_partition_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8.8, 5.8))
bars = ax.bar(core1a_attribute_aggregate_summary['attribute_group'], core1a_attribute_aggregate_summary['mean_forget_drop_hit_pp'], color=[ATTRIBUTE_COLORS[g] for g in core1a_attribute_aggregate_summary['attribute_group']], width=0.65)
ax.set_title('CORE 1a - Single Attribute Group Effect (Average)', loc='left', pad=14)
ax.set_ylabel('Average Forget Drop Hit@10 (pp)')
ax.set_xlabel('Attribute group')
ax.set_ylim(0, core1a_attribute_aggregate_summary['mean_forget_drop_hit_pp'].max() * 1.18)
add_bar_labels(ax, bars)
fig.tight_layout()
fig.savefig(FIG_DIR / 'core1a_attribute_group_effects.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

heatmap_df = (
    ugp_results_k10[['method', 'setting_display', 'attribute_group', 'forget_drop_hit_pp']]
    .copy()
)
group_rank = {'Gender': 0, 'Age': 1, 'Occupation': 2}
setting_order = (
    heatmap_df[['setting_display', 'attribute_group']]
    .drop_duplicates()
    .assign(group_rank=lambda df: df['attribute_group'].map(group_rank))
    .sort_values(['group_rank', 'setting_display'])['setting_display']
    .tolist()
)
pivot = heatmap_df.pivot(index='method', columns='setting_display', values='forget_drop_hit_pp').reindex(index=method_order, columns=setting_order)
fig, ax = plt.subplots(figsize=(18, 4.8))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGnBu')
ax.set_title('CORE 1a - Single Attribute Partition Heatmap (Forget Drop Hit@10)', loc='left', pad=14)
ax.set_yticks(np.arange(len(method_order)))
ax.set_yticklabels([METHOD_LABELS[m] for m in method_order])
ax.set_xticks(np.arange(len(setting_order)))
ax.set_xticklabels(setting_order, rotation=70, ha='right', rotation_mode='anchor', fontsize=9)
cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('Forget Drop Hit@10 (pp)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'core1a_single_attribute_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

print('CORE 1a selesai. Tables dan figures sudah disimpan.')


In [ ]:
sim_env = prepare_similarity_environment()
ratings_df = sim_env['ratings_df']
movies_df = sim_env['movies_df']
users_df = sim_env['users_df']
sample_users = sim_env['sample_users']
candidate_movies = sim_env['candidate_movies']
trajectories_all = sim_env['trajectories_all']
user_states = sim_env['user_states']
state_dim = sim_env['state_dim']
build_state_fn = sim_env['build_state_fn']
num_actions = len(candidate_movies)
trajectory_by_user = {traj['user_id']: traj for traj in trajectories_all}

def run_similarity_scenario(family_label, row):
    method = row['method']
    if family_label == 'Random':
        forget_users, retain_users = build_random_split(sample_users)
        scenario_label = 'Random'
    elif family_label == 'Profile-Based Partition':
        forget_users, retain_users = build_profile_split(users_df, sample_users)
        scenario_label = 'Profile-Based Partition'
    elif family_label == 'Single Attribute Partition':
        forget_users, retain_users = build_single_attribute_split(users_df, sample_users, row['setting_type'], row['setting_value_raw'])
        scenario_label = f"Single Attribute Partition - {row['setting_display']}"
    else:
        raise ValueError(f'Unsupported family_label: {family_label}')

    forget_set = set(forget_users.tolist())
    retain_set = set(retain_users.tolist())
    forget_trajectories = [trajectory_by_user[uid] for uid in forget_users if uid in trajectory_by_user]
    retain_trajectories = [trajectory_by_user[uid] for uid in retain_users if uid in trajectory_by_user]
    forget_state_map = {uid: user_states[uid] for uid in forget_users if uid in user_states}
    retain_state_map = {uid: user_states[uid] for uid in retain_users if uid in user_states}

    net_base = load_policy(row['trained_model_path'], state_dim, num_actions, row['hidden_dim'])
    net_ul = load_policy(row['unlearned_model_path'], state_dim, num_actions, row['hidden_dim'])

    r_before = evaluate_per_user(net_base, retain_trajectories, candidate_movies, build_state_fn, K=K_TARGET)
    r_after = evaluate_per_user(net_ul, retain_trajectories, candidate_movies, build_state_fn, K=K_TARGET)
    f_before = evaluate_per_user(net_base, forget_trajectories, candidate_movies, build_state_fn, K=K_TARGET)
    f_after = evaluate_per_user(net_ul, forget_trajectories, candidate_movies, build_state_fn, K=K_TARGET)

    df_r = r_before.merge(r_after, on='uid', suffixes=('_before', '_after'))
    df_r['delta'] = df_r['hit_after'] - df_r['hit_before']
    df_r['damaged'] = df_r['delta'] < 0

    df_f = f_before.merge(f_after, on='uid', suffixes=('_before', '_after'))
    df_f['delta'] = df_f['hit_after'] - df_f['hit_before']
    n_core = max(1, int(math.ceil(len(df_f) * 0.10))) if len(df_f) else 0
    core_uids = df_f.sort_values('delta').head(n_core)['uid'].tolist() if n_core else []

    stats_rows = compute_cosine_similarity_stats(df_r, core_uids, retain_state_map, forget_state_map)
    for item in stats_rows:
        item.update({
            'partition_family': family_label,
            'scenario_label': scenario_label,
            'method': method,
            'method_label': METHOD_LABELS.get(method, method),
            'setting_display': row.get('setting_display', f'{family_label} 1%'),
            'core_forget_count': len(core_uids),
            'forget_user_count': len(forget_trajectories),
            'retain_user_count': len(retain_trajectories),
        })
    del net_base, net_ul
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return stats_rows

similarity_rows = []
for _, row in selected_random_df.iterrows():
    similarity_rows.extend(run_similarity_scenario('Random', row))
for _, row in selected_profile_df.iterrows():
    similarity_rows.extend(run_similarity_scenario('Profile-Based Partition', row))
for _, row in selected_ugp_best_df.iterrows():
    similarity_rows.extend(run_similarity_scenario('Single Attribute Partition', row))

core1b_similarity_stats = pd.DataFrame(similarity_rows)
cross_partition_summary = (
    core1b_similarity_stats
    .groupby(['partition_family', 'proximity_type'], as_index=False)
    .agg(
        mean_delta_mean=('Delta Mean', 'mean'),
        mean_damaged=('Mean (Damaged)', 'mean'),
        mean_ok=('Mean (OK)', 'mean'),
        median_p_value=('p-value', 'median'),
        mean_pearson_r=('Pearson r', 'mean'),
    )
)
cross_partition_summary['partition_family'] = cross_partition_summary['partition_family'] + ' - Summary'
cross_partition_summary['scenario_label'] = 'Cross-partition summary'
cross_partition_summary['method'] = 'ALL'
cross_partition_summary['method_label'] = 'All methods'
cross_partition_summary['setting_display'] = 'Aggregate'
cross_partition_summary['core_forget_count'] = np.nan
cross_partition_summary['forget_user_count'] = np.nan
cross_partition_summary['retain_user_count'] = np.nan
cross_partition_summary['N Damaged'] = np.nan
cross_partition_summary['N OK'] = np.nan
cross_partition_summary['U Statistic'] = np.nan
cross_partition_summary['p-value'] = cross_partition_summary['median_p_value']
cross_partition_summary['Pearson p'] = np.nan
cross_partition_summary = cross_partition_summary.rename(columns={
    'mean_delta_mean': 'Delta Mean',
    'mean_damaged': 'Mean (Damaged)',
    'mean_ok': 'Mean (OK)',
    'mean_pearson_r': 'Pearson r',
})
core1b_similarity_stats = pd.concat([core1b_similarity_stats, cross_partition_summary[core1b_similarity_stats.columns]], ignore_index=True)
core1b_similarity_stats.to_csv(TABLE_DIR / 'core1b_similarity_stats.csv', index=False)

display(core1b_similarity_stats.sort_values(['partition_family', 'method_label', 'proximity_type']))

plot_df = core1b_similarity_stats[core1b_similarity_stats['method'] != 'ALL'].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4), sharey=True)
for ax, proximity in zip(axes, ['Centroid', 'Pairwise']):
    subset = plot_df[plot_df['proximity_type'] == proximity].copy()
    method_order = TARGET_METHODS
    family_order = ['Random', 'Profile-Based Partition', 'Single Attribute Partition']
    x = np.arange(len(method_order))
    width = 0.24
    for idx, family in enumerate(family_order):
        fam_df = subset[subset['partition_family'] == family].set_index('method').reindex(method_order).reset_index()
        bars = ax.bar(x + (idx - 1) * width, fam_df['Delta Mean'], width=width, color=PARTITION_COLORS[family], label=family)
        add_bar_labels(ax, bars)
    ax.set_title(f'{proximity} proximity')
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS[m] for m in method_order])
    ax.set_xlabel('Method')
axes[0].set_ylabel('Delta Mean (Damaged - OK)')
fig.suptitle('CORE 1b - Similarity Overview', x=0.06, ha='left', y=1.02, fontsize=15)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=3, bbox_to_anchor=(0.5, 1.03), loc='lower center')
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(FIG_DIR / 'core1b_similarity_overview.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

print('CORE 1b selesai. Similarity stats dan figure sudah disimpan.')
